# Evaluación

El conjunto de test se usa **una sola vez**, con el modelo ya elegido. Este capítulo
responde a: ¿cuánto se equivoca el modelo, dónde se equivoca, y para qué moléculas se
puede confiar en él?

In [ ]:
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import spearmanr, pearsonr

sns.set_theme(style="whitegrid")
datos = Path("../datos/procesados")

paquete = joblib.load("../modelos/modelo_final.joblib")
modelo, scaler, idx_te = paquete["modelo"], paquete["scaler"], paquete["idx_test"]

X_fp = np.load(datos / "X_fp.npy")
X_desc = pd.read_parquet(datos / "X_desc.parquet")
meta = pd.read_parquet(datos / "y_meta.parquet")

X = np.hstack([X_fp, scaler.transform(X_desc)]).astype(np.float32)
X_te, y_te = X[idx_te], meta["pic50"].to_numpy()[idx_te]
y_pred = modelo.predict(X_te)
len(y_te)

## Métricas globales

In [ ]:
rmse = np.sqrt(mean_squared_error(y_te, y_pred))
mae = mean_absolute_error(y_te, y_pred)
r2 = r2_score(y_te, y_pred)
rho = spearmanr(y_te, y_pred).statistic
r_p = pearsonr(y_te, y_pred).statistic

print(f"RMSE      {rmse:.3f} unidades log")
print(f"MAE       {mae:.3f}")
print(f"R²        {r2:.3f}")
print(f"Spearman  {rho:.3f}")
print(f"Pearson   {r_p:.3f}")
print(f"\n% predicciones con error < 0,5 log: {(np.abs(y_te-y_pred) < 0.5).mean():.1%}")
print(f"% predicciones con error < 1,0 log: {(np.abs(y_te-y_pred) < 1.0).mean():.1%}")

Un error de 0,5 unidades log significa un factor ~3 en la concentración; un error de
1,0 significa un factor 10. Para priorizar compuestos a sintetizar, un RMSE de 0,7 con
buena correlación de rangos ya es útil.

## Predicho frente a observado

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

lim = [min(y_te.min(), y_pred.min()) - 0.3, max(y_te.max(), y_pred.max()) + 0.3]
axes[0].scatter(y_te, y_pred, s=14, alpha=0.5)
axes[0].plot(lim, lim, "k--", lw=1, label="y = x")
axes[0].fill_between(lim, [v-1 for v in lim], [v+1 for v in lim],
                     color="gray", alpha=0.12, label="±1 log")
axes[0].set_xlabel("pIC50 observado")
axes[0].set_ylabel("pIC50 predicho")
axes[0].set_title(f"Test (scaffold split) — R² = {r2:.2f}")
axes[0].legend()

residuos = y_te - y_pred
axes[1].scatter(y_pred, residuos, s=14, alpha=0.5)
axes[1].axhline(0, color="k", ls="--", lw=1)
axes[1].set_xlabel("pIC50 predicho")
axes[1].set_ylabel("residuo (obs - pred)")
axes[1].set_title("Residuos")
plt.tight_layout()

Lo que se busca en los residuos:

- **Forma de embudo** → heterocedasticidad; el modelo es menos fiable en un extremo.
- **Pendiente negativa** → *regresión a la media*: el modelo comprime el rango,
  subestima los muy potentes y sobreestima los débiles. Es el patrón más común y hay que
  reportarlo, porque afecta directamente al cribado de los compuestos más interesantes.
- **Grupos de residuos grandes** → suelen ser un scaffold entero fuera de dominio.

In [ ]:
# Rendimiento por tramos de potencia
tramos = pd.cut(y_te, bins=[0, 5, 6, 7, 8, 9, 15],
                labels=["<5", "5-6", "6-7", "7-8", "8-9", ">9"])
res = pd.DataFrame({"obs": y_te, "pred": y_pred, "tramo": tramos})

tabla = res.groupby("tramo", observed=True).apply(
    lambda g: pd.Series({
        "n": len(g),
        "RMSE": np.sqrt(mean_squared_error(g["obs"], g["pred"])),
        "sesgo": (g["pred"] - g["obs"]).mean(),
    })
)
tabla

## Dominio de aplicabilidad

Un modelo QSAR solo es fiable para moléculas químicamente parecidas a las de
entrenamiento. Se cuantifica con la similitud de Tanimoto al vecino más cercano del
conjunto de entrenamiento.

In [ ]:
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator

gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

idx_tr = np.setdiff1d(np.arange(len(meta)), idx_te)
fp_tr = [gen.GetFingerprint(Chem.MolFromSmiles(s))
         for s in meta["smiles"].iloc[idx_tr]]
fp_te = [gen.GetFingerprint(Chem.MolFromSmiles(s))
         for s in meta["smiles"].iloc[idx_te]]

sim_max = np.array([max(DataStructs.BulkTanimotoSimilarity(f, fp_tr)) for f in fp_te])

plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1)
sns.histplot(sim_max, bins=40)
plt.xlabel("Tanimoto al vecino más cercano en train")
plt.title("Distancia al dominio de entrenamiento")

plt.subplot(1, 2, 2)
plt.scatter(sim_max, np.abs(residuos), s=14, alpha=0.5)
plt.xlabel("similitud al vecino más cercano")
plt.ylabel("|error|")
plt.title("El error crece al alejarse del dominio")
plt.tight_layout()

print(f"correlación similitud vs |error|: {spearmanr(sim_max, np.abs(residuos)).statistic:.3f}")

In [ ]:
# Error por franja de similitud: base para declarar el dominio de aplicabilidad
franjas = pd.cut(sim_max, bins=[0, 0.3, 0.4, 0.5, 0.7, 1.0])
pd.DataFrame({"sim": franjas, "obs": y_te, "pred": y_pred}).groupby(
    "sim", observed=True).apply(
    lambda g: pd.Series({"n": len(g),
                         "RMSE": np.sqrt(mean_squared_error(g["obs"], g["pred"]))})
)

```{tip}
Esta tabla permite acompañar cada predicción futura de una advertencia honesta:
«Tanimoto 0,25 al dominio de entrenamiento; RMSE esperado en esa franja: 1,1».
Es la diferencia entre un modelo usable y un número sin contexto.
```

## Comparación con partición aleatoria

Reportar las dos cifras demuestra que se entiende el sesgo optimista de la partición
aleatoria, y es un punto fuerte del trabajo, no una debilidad.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.base import clone

y_all = meta["pic50"].to_numpy()
Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(X, y_all, test_size=0.15, random_state=42)

m_rand = clone(modelo).fit(Xa_tr, ya_tr)
pred_rand = m_rand.predict(Xa_te)

comp = pd.DataFrame([
    {"partición": "scaffold (honesta)",
     "RMSE": rmse, "R2": r2, "Spearman": rho},
    {"partición": "aleatoria (optimista)",
     "RMSE": np.sqrt(mean_squared_error(ya_te, pred_rand)),
     "R2": r2_score(ya_te, pred_rand),
     "Spearman": spearmanr(ya_te, pred_rand).statistic},
])
comp

## Peores predicciones

Mirar los diez mayores errores a menudo revela problemas de datos (un valor mal
transcrito, un compuesto con mecanismo distinto) más que fallos del modelo.

In [ ]:
peores = pd.DataFrame({
    "chembl_id": meta["chembl_id"].iloc[idx_te].values,
    "smiles": meta["smiles"].iloc[idx_te].values,
    "obs": y_te, "pred": y_pred, "error": residuos,
    "sim_max": sim_max,
}).assign(abs_err=lambda d: d["error"].abs()).nlargest(10, "abs_err")

peores[["chembl_id", "obs", "pred", "error", "sim_max"]]

In [ ]:
from rdkit.Chem import Draw

ms = [Chem.MolFromSmiles(s) for s in peores["smiles"].head(6)]
leyendas = [f"obs {o:.1f} / pred {p:.1f}"
            for o, p in zip(peores["obs"].head(6), peores["pred"].head(6))]
Draw.MolsToGridImage(ms, molsPerRow=3, subImgSize=(280, 220), legends=leyendas)